In [1]:
import math


import networkx as nx
from netbone.utils.utils import edge_properties


def fraction_filter(backbone, value, narrate=True, secondary_property='weight', secondary_property_ascending=False,
                    **kwargs):
    data = backbone.to_dataframe()
    filter_by = [backbone.property_name]
    ascending = [backbone.ascending]


    if backbone.filter_on == 'Edges':
        filter_by.append(secondary_property)
        ascending.append(secondary_property_ascending)


    if fraction_filter in backbone.compatible_filters():
        # Sort by primary (backbone.property_name) and optional secondary property
        data = data.sort_values(by=filter_by, ascending=ascending)


        if narrate:
            backbone.narrate()


        if backbone.filter_on == 'Edges':
            # Number of edges to target by fraction
            target_m = math.ceil(value * len(data))
            # Build full graph of candidate edges (sorted already)
            G_full = nx.from_pandas_edgelist(data, edge_attr=edge_properties(data))


            # Ensure we operate over the same node set as the backbone graph when available
            # (helps keep isolated nodes, if any)
            if hasattr(backbone, 'graph') and backbone.graph is not None:
                for n in backbone.graph.nodes:
                    if n not in G_full:
                        G_full.add_node(n)


            # If there are no nodes or no edges, just return empty graph
            if G_full.number_of_nodes() == 0 or G_full.number_of_edges() == 0:
                return G_full


            # If the underlying candidate graph isn't connected, we cannot produce a
            # fully connected backbone without adding non-candidate edges.
            # In that case, fall back to the original fraction behavior (with a warning).
            try:
                is_conn = nx.is_connected(G_full)
            except nx.NetworkXPointlessConcept:
                is_conn = False


            if not is_conn:
                # Fallback: original behavior (may be disconnected)
                target_m = max(0, min(target_m, len(data)))
                return nx.from_pandas_edgelist(data[:target_m], edge_attr=edge_properties(data))


            # Ensure at least a spanning tree: a connected simple graph needs n-1 edges
            n = G_full.number_of_nodes()
            min_required = max(0, n - 1)
            target_m = max(target_m, min_required)
            target_m = min(target_m, G_full.number_of_edges())


            # Start from the maximum spanning tree for strong connectivity with best scores
            # Use the backbone's property as the weight key (e.g., 'score')
            T = nx.maximum_spanning_tree(G_full, weight=backbone.property_name)


            # If fraction requests no more than a tree, we're done
            if target_m <= T.number_of_edges():
                # Ensure all nodes are present
                H = nx.Graph()
                H.add_nodes_from(G_full.nodes())
                H.add_edges_from(T.edges(data=True))
                return H


            # Otherwise, add highest-ranked remaining edges until target_m is reached
            H = nx.Graph()
            H.add_nodes_from(G_full.nodes())
            H.add_edges_from(T.edges(data=True))


            # Iterate over sorted edges and add those not already in H
            # Build a quick lookup for attributes from G_full
            for _, row in data.iterrows():
                if H.number_of_edges() >= target_m:
                    break
                u = row['source']
                v = row['target']
                if not H.has_edge(u, v) and G_full.has_edge(u, v):
                    # Preserve attributes from G_full
                    attr = G_full.get_edge_data(u, v) or {}
                    H.add_edge(u, v, **attr)


            return H
        else:
            # Node-based fraction: preserve original behavior
            value = math.ceil(value * len(backbone.graph))
            return backbone.graph.subgraph(list(data[:value].index)).copy()


    print("The accepted filters for " + backbone.method_name + " are: " + ', '.join(
        [fun.__name__ for fun in backbone.compatible_filters()]))

In [2]:
def doubly_stochastic_fast_uf(data):
    import warnings
    import time
    import numpy as np
    import pandas as pd
    import networkx as nx
    from netbone.backbone import Backbone
    from netbone.filters import boolean_filter, threshold_filter

    # ---------- Timing ----------
    t0 = time.perf_counter()

    # ---------- Input handling ----------
    t = time.perf_counter()
    if isinstance(data, pd.DataFrame):
        table = data.copy()
    elif isinstance(data, nx.Graph):
        table = nx.to_pandas_edgelist(data)
    else:
        print("data should be a pandas dataframe or nx graph")
        return
    table2 = table.copy()
    print(f"[TIME] Input handling: {time.perf_counter() - t:.4f}s")

    # ---------- Pivot ----------
    t = time.perf_counter()
    table = pd.pivot_table(table, values="weight", index="source", columns="target",
                            aggfunc="sum", fill_value=0) + 0.0001
    print(f"[TIME] Pivot table: {time.perf_counter() - t:.4f}s")

    # ---------- Sinkhorn ----------
    t = time.perf_counter()
    row_sums = table.sum(axis=1)
    attempts = 0
    while np.std(row_sums) > 1e-12:
        table = table.div(row_sums, axis=0)
        col_sums = table.sum(axis=0)
        table = table.div(col_sums, axis=1)
        row_sums = table.sum(axis=1)
        attempts += 1
        if attempts > 1000:
            warnings.warn(
                "Matrix could not be reduced to doubly stochastic. See Sec. 3 of Sinkhorn 1964",
                RuntimeWarning
            )
            return pd.DataFrame()
    print(f"[TIME] Sinkhorn normalization ({attempts} iters): {time.perf_counter() - t:.4f}s")

    # ---------- Melt & sort ----------
    t = time.perf_counter()
    table = pd.melt(table.reset_index(), id_vars="source")
    table = table[table["source"] < table["target"]]
    table = table[table["value"] > 0]
    table = table.sort_values(by="value", ascending=False)
    print(f"[TIME] Melt & sort: {time.perf_counter() - t:.4f}s")

    # ---------- Merge original weights ----------
    t = time.perf_counter()
    table = table.merge(table2[["source", "target", "weight"]], on=["source", "target"])
    print(f"[TIME] Merge original weights: {time.perf_counter() - t:.4f}s")


    # ---------- Final formatting ----------
    t = time.perf_counter()
    table = table[table["value"] >= 0].rename(columns={"value": "score"}).fillna(False)
    table = table[table["source"] <= table["target"]]

    backbone = Backbone(
        nx.from_pandas_edgelist(table, edge_attr=["weight", "score"]),
        method_name="Doubly Stochastic Filter",
        property_name="score",
        ascending=False,
        compatible_filters=[boolean_filter, threshold_filter, fraction_filter],
        filter_on="Edges"
    )
    print(f"[TIME] Final formatting & Backbone creation: {time.perf_counter() - t:.4f}s")
    print(f"[TIME] TOTAL: {time.perf_counter() - t0:.4f}s")

    return backbone


In [3]:
import netbone as nb
import networkx as nx

In [4]:
from networkx.drawing.nx_agraph import read_dot

# Load graph directly
g = read_dot("../graph/subgraph_0.dot")

# Convert 'weight' edge attributes from str to float (in-place)
converted = 0
total_with_weight = 0

for uu, vv, kk, data in g.edges(keys=True, data=True):
    if 'weight' in data:
        total_with_weight += 1
        w = data['weight']
        if isinstance(w, str):
            try:
                data['weight'] = float(w)
                converted += 1
            except ValueError:
                pass  # leave as-is if not a valid float

import pandas as pd

def _to_int(n):
    try:
        return int(n)
    except Exception:
        return n

rows = []
for u, v, data in g.edges(data=True):
    ui = _to_int(u)
    vi = _to_int(v)
    w = data.get("weight", 1.0)
    if isinstance(w, str):
        try:
            w = float(w)
        except Exception:
            continue
    a, b = (ui, vi) if ui <= vi else (vi, ui)
    rows.append({"source": a, "target": b, "weight": w})

# Aggregate any duplicates after canonicalization
df = pd.DataFrame(rows).groupby(["source", "target"], as_index=False).agg(weight=("weight", "sum"))


In [5]:
print(len(df))

6508


In [6]:
# # apply the choosen backbone extraction method
# b = nb.doubly_stochastic(g)

# # extract the backbone based on the default threshold
# backbone1 = boolean_filter(b)

# # extract the backbone based on a threshold(0.7)
# backbone2 = threshold_filter(b, 0.7)

# # extract the backbone keeping a fraction of edges(0.15)
# backbone3 = fraction_filter(b, 0.15)


In [ ]:
# apply the choosen backbone extraction method
b = nb.pmfg(df)
#b = nb.doubly_stochastic(df)
# # extract the backbone based on the default threshold
backbone = fraction_filter(b, 0.50)

# print("Number of links:", backbone1.number_of_edges())

[TIME] Input handling: 0.0005s
[TIME] Pivot table: 0.0600s
[TIME] Sinkhorn normalization (32 iters): 0.8619s
[TIME] Melt & sort: 0.6833s
[TIME] Merge original weights: 0.4658s
[TIME] Final formatting & Backbone creation: 0.0106s
[TIME] TOTAL: 2.0825s
Doubly Stochastic Filter


In [8]:
# # extract the backbone based on the default threshold
backbone = fraction_filter(b, 0.50)

Doubly Stochastic Filter


In [9]:
print(backbone.number_of_edges())
print(len(b.to_dataframe()))

3254
6508


In [10]:
from networkx.drawing.nx_agraph import write_dot

out_path = "../graph/backbone.dot"
write_dot(backbone, out_path)
print(f"Wrote DOT to {out_path} ({backbone.number_of_nodes()} nodes, {backbone.number_of_edges()} edges)")

Wrote DOT to ../graph/backbone.dot (2682 nodes, 3254 edges)


In [11]:
# # Connectivity check for the computed backbone
# import networkx as nx


# if 'backbone' not in globals():
#     print("Variable 'backbone' not found. Please run the backbone-building cells first.")
# else:
#     G = backbone
#     try:
#         connected = nx.is_connected(G)
#     except nx.NetworkXPointlessConcept:
#         connected = (G.number_of_nodes() <= 1)
#     print(f"Connected: {connected}")
#     if connected:
#         print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
#     else:
#         try:
#             import itertools
#             comps = list(nx.connected_components(G))
#             sizes = [len(c) for c in comps]
#             sizes_sorted = sorted(sizes, reverse=True)
#             print(f"Components: {len(comps)} | Largest sizes: {sizes_sorted[:5]}")
#         except Exception as e:
#             print(f"Could not compute components: {e}")